# Actividad 06: Slowly Changing Dimensions (SCD)

Rebecca Chambers va a ser ascendida. ¿Cómo rastreamos este cambio en nuestro Data Warehouse?

In [ ]:
import duckdb

con = duckdb.connect(':memory:')

# Setup
con.execute("""
    CREATE TABLE researcher_dim (
        researcher_id   INTEGER PRIMARY KEY,
        researcher_name VARCHAR,
        position        VARCHAR,
        department      VARCHAR,
        hire_date       DATE,
        country         VARCHAR
    );
""")

con.execute("""
    INSERT INTO researcher_dim VALUES
    (1, 'Rebecca Chambers', 'Junior Scientist', 'Bioresearch', DATE '2023-09-01', 'USA'),
    (2, 'Jill Valentine',   'Lead Analyst',     'Analytics',   DATE '2022-04-15', 'USA'),
    (3, 'Chris Redfield',   'Data Engineer',    'Platform',    DATE '2021-01-10', 'USA'),
    (4, 'Albert Wesker',    'Director',         'Executive',   DATE '2018-06-01', 'Germany');
""")

con.execute("""
    CREATE TABLE experiment_fact (
        experiment_id   INTEGER PRIMARY KEY,
        date_id         INTEGER,
        researcher_id   INTEGER,
        experiment_cost DECIMAL(12,2)
    );
""")

con.execute("""
    INSERT INTO experiment_fact VALUES
    (9001, 20250115, 1, 1200.00),
    (9002, 20250220, 1, 1500.00),
    (9003, 20260105, 1, 1700.00),
    (9004, 20260410, 1, 1800.00);
""")

print('Initial state:')
con.execute('SELECT * FROM researcher_dim').show()
con.execute('SELECT * FROM experiment_fact').show()

## El evento

El **2026-04-01**, Rebecca Chambers es ascendida:
- **De:** Junior Scientist, Bioresearch
- **A:** Lead Scientist, Bio-Weapons Research

---

> **Recuerda:** **SCD Type I** sobrescribe (pierde historial); **Type II** añade una fila nueva con ventana de validez (historial completo); **Type III** guarda el valor anterior en columnas aparte.

### Los tres tipos de SCD

**Type I** sobrescribe el valor, **Type II** guarda cada versión con su ventana de validez y **Type III** conserva el valor anterior en columnas aparte.

```mermaid
flowchart TB
    subgraph T1["Type I: sobrescribe"]
        A1["antes: Junior Scientist"] --> A2["después: Lead Scientist<br/>1 sola fila, historial perdido"]
    end
    subgraph T2["Type II: añade fila nueva"]
        B1["fila 1: Junior<br/>2023-09-01 a 2026-04-01"]
        B2["fila 2: Lead<br/>desde 2026-04-01"]
    end
    subgraph T3["Type III: columna con el valor anterior"]
        C1["position: Lead<br/>previous_position: Junior"]
    end
```


## Tarea 1: Type I - sobrescribir

Sobrescribe la fila de Rebecca con sus nuevos valores.

In [ ]:
# TODO: Sobrescribe la fila de Rebecca



In [ ]:
# Ejecuta la consulta analitica
result = con.execute("""
    SELECT r.position, SUM(f.experiment_cost) AS total_cost
    FROM experiment_fact f
    JOIN researcher_dim r ON r.researcher_id = f.researcher_id
    WHERE f.researcher_id = 1
    GROUP BY r.position
""").fetchall()
print('Resultado Type I:')
for row in result:
    print(row)

**Pregunta:** Cuanto coste de *Junior Scientist* aparece? Es historicamente correcto?

## Tarea 2: Type II - nueva fila + ventana de validez

Conserva la fila antigua y añade una nueva con fechas de validez.

### El esquema de una dimensión Type II

Cada versión es una **fila** con su propia clave (`researcher_sk`); la ventana de validez indica desde cuándo hasta cuándo vale esa versión.

```mermaid
erDiagram
    RESEARCHER_DIM ||--o{ EXPERIMENT_FACT : FK-researcher
    RESEARCHER_DIM {
        int researcher_sk PK
        int researcher_id
        string researcher_name
        string position
        string department
        date valid_from
        date valid_to
        bool is_current
    }
    EXPERIMENT_FACT {
        int experiment_id PK
        int researcher_id FK
        int date_id FK
        float experiment_cost
    }
```


### Las ventanas de validez de Rebecca

La fila antigua se **cierra** el día del cambio y la nueva queda como vigente.

```mermaid
gantt
    title Ventanas de validez de researcher_dim (Type II)
    dateFormat YYYY-MM-DD
    axisFormat %Y-%m
    section researcher_id = 1
    Junior Scientist :done, 2023-09-01, 2026-04-01
    Lead Scientist :active, 2026-04-01, 2026-12-31
```


In [ ]:
# Reinicia researcher_dim para Type II
con.execute('DROP TABLE researcher_dim')

# Crea con columnas SCD Type II
con.execute("""
    CREATE TABLE researcher_dim (
        researcher_sk   INTEGER PRIMARY KEY,
        researcher_id   INTEGER,
        researcher_name VARCHAR,
        position        VARCHAR,
        department      VARCHAR,
        hire_date       DATE,
        country         VARCHAR,
        valid_from      DATE,
        valid_to        DATE,
        is_current      BOOLEAN
    );
""")

In [ ]:
# Implementa Type II: anade nueva fila con ventana de validez
con.execute("""
    INSERT INTO researcher_dim VALUES
    (1, 1, 'Rebecca Chambers', 'Junior Scientist', 'Bioresearch', '2023-09-01', 'USA', '2023-09-01', '2026-04-01', false),
    (2, 1, 'Rebecca Chambers', 'Lead Scientist', 'Bio-Weapons Research', '2023-09-01', 'USA', '2026-04-01', NULL, true),
    (3, 2, 'Jill Valentine', 'Senior Scientist', 'Advanced Research', '2020-03-15', 'USA', '2020-03-15', NULL, true),
    (4, 3, 'Chris Redfield', 'Data Engineer', 'IT', '2019-06-01', 'UK', '2019-06-01', NULL, true),
    (5, 4, 'Albert Wesker', 'Director', 'Executive', '2015-01-10', 'USA', '2015-01-10', NULL, true);
""")

print('Historial Type II:')
for row in con.execute("SELECT researcher_name, position, valid_from, valid_to, is_current FROM researcher_dim WHERE researcher_id = 1 ORDER BY valid_from").fetchall():
    print(row)

In [ ]:
# Consulta la posicion actual
result = con.execute("""
    SELECT researcher_name, position, valid_from, is_current
    FROM researcher_dim
    WHERE researcher_id = 1
    ORDER BY valid_from
""").fetchall()
print('Historial Type II:')
for row in result:
    print(row)

**Pregunta:** Cuanto coste se atribuye a *Junior Scientist* vs *Lead Scientist*?

## Tarea 3: Type III - columna de valor anterior

Añade columnas para almacenar el valor anterior.

### Tarea 3: Aplica Type III

Pasos:
1. Elimina y recrea `researcher_dim` con columnas `previous_position` y `previous_department`
2. Inserta los datos iniciales (igual que antes)
3. Escribe un UPDATE para mover los valores actuales a `previous_*`

In [ ]:
# Type III: Columna de valor anterior
con.execute('DROP TABLE researcher_dim')

con.execute("""
    CREATE TABLE researcher_dim (
        researcher_id INTEGER PRIMARY KEY,
        researcher_name VARCHAR,
        position VARCHAR,
        department VARCHAR,
        previous_position VARCHAR,
        previous_department VARCHAR,
        hire_date DATE,
        country VARCHAR
    );
""")

con.execute("""
    INSERT INTO researcher_dim VALUES
    (1, 'Rebecca Chambers', 'Junior Scientist', 'Bioresearch', NULL, NULL, '2023-09-01', 'USA'),
    (2, 'Jill Valentine', 'Senior Scientist', 'Advanced Research', NULL, NULL, '2020-03-15', 'USA'),
    (3, 'Chris Redfield', 'Data Engineer', 'IT', NULL, NULL, '2019-06-01', 'UK'),
    (4, 'Albert Wesker', 'Director', 'Executive', NULL, NULL, '2015-01-10', 'USA');
""")

con.execute("""
    UPDATE researcher_dim
    SET previous_position = position,
        previous_department = department,
        position = 'Lead Scientist',
        department = 'Bio-Weapons Research'
    WHERE researcher_id = 1;
""")

print('Type III:')
for row in con.execute("SELECT researcher_name, position, previous_position, department, previous_department FROM researcher_dim WHERE researcher_id = 1").fetchall():
    print(row)

**Pregunta:** Puedes responder "Cuanto costo Rebecca mientras era Junior Scientist" con Type III?

---

## Tarea 4: Decide

Que estrategia SCD recomiendas para `researcher_dim` de Parasol Corp y por que?

In [ ]:
# TODO: Escribe tu recomendacion
scd_recommendation = ''   # 'Type I', 'Type II', o 'Type III'
scd_justification = ''    # 2-3 frases explicando por que



---

## Guarda tu trabajo

Copia tu solucion en `scripts/scd.py`.

El fichero debe ser un script de Python independiente que:
1. Importa `duckdb`
2. Crea la conexion `con = duckdb.connect(":memory:")`
3. Contiene las tablas y los ejercicios SCD que has completado
4. Define las variables `scd_recommendation` y `scd_justification`

```bash
pytest tests/ -v
```


---

## ¡Curso completado!

Has terminado las actividades del curso de Data Warehouse: modelado OLTP, normalización, Data Marts, esquema estrella, consultas analíticas y SCD.